# 02 — Jamba Architecture Lab

Prototype and explore the **Jamba** architecture using the building blocks in
`src/core/`.  This is where you iterate before promoting to `src/`.

In [1]:
import sys
sys.path.insert(0, '../..')

import torch
from src.models.jamba.config import JambaConfig
from src.models.jamba.model import JambaSLM
from src.core.generation import generate
from src.utils.training import count_params

In [2]:
# ── Build a tiny model for fast iteration ────────────────────────────────
tiny_cfg = JambaConfig(
    vocab_size=50257, block_size=32,
    n_layer=4, n_embd=64, n_head=2,
    mamba_d_state=16, mamba_d_conv=4, mamba_expand=2,
    intermediate_size=128, dropout=0.0,
)
model = JambaSLM(tiny_cfg)
print(f'Parameters: {count_params(model):,}')

Parameters: 3,411,840


In [3]:
# ── Verify forward pass shapes ────────────────────────────────────────────
B, T = 2, tiny_cfg.block_size
idx = torch.randint(0, tiny_cfg.vocab_size, (B, T))
targets = torch.randint(0, tiny_cfg.vocab_size, (B, T))

logits, loss = model(idx, targets)
print(f'logits: {logits.shape}  |  loss: {loss.item():.4f}')

logits: torch.Size([2, 32, 50257])  |  loss: 10.8347


In [4]:
# ── Test generation ───────────────────────────────────────────────────────
prompt = torch.randint(0, tiny_cfg.vocab_size, (1, 4))
output = generate(model, prompt, max_new_tokens=10, temperature=1.0, top_k=50)
print(f'Input tokens:  {prompt[0].tolist()}')
print(f'Output tokens: {output[0].tolist()}')

Input tokens:  [419, 21581, 30798, 49043]
Output tokens: [419, 21581, 30798, 49043, 14976, 14976, 5786, 27822, 43939, 7468, 43939, 21588, 9801, 9801]


In [5]:
# ── Parameter count comparison across model sizes ────────────────────────
configs = {
    'jamba_tiny':   JambaConfig(vocab_size=50257, block_size=16,  n_layer=4,  n_embd=64,  n_head=2,  intermediate_size=128),
    'jamba_small':  JambaConfig(vocab_size=50257, block_size=128, n_layer=8,  n_embd=384, n_head=6,  intermediate_size=1024),
    'jamba_medium': JambaConfig(vocab_size=50257, block_size=256, n_layer=16, n_embd=768, n_head=12, intermediate_size=2048),
}
for name, cfg in configs.items():
    n = count_params(JambaSLM(cfg))
    print(f'{name:14s}: {n:>12,} params  ({n/1e6:.1f}M)')

jamba_tiny    :    3,411,840 params  (3.4M)
jamba_small   :   34,815,744 params  (34.8M)
jamba_medium  :  162,006,528 params  (162.0M)
